# L13a: Synthetic Markets with Jump Hidden Markov Models
In this lecture, we build synthetic markets: simulated price histories that share the features of real ones, such as fat tails and volatility clustering. A backtest tests a trading method on one historical path. How would the same method do on thousands of plausible paths, including sell-offs that never happened?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Construct Markov chains and hidden Markov models:__ Define a state space and a transition matrix, compute the stationary distribution, and describe how hidden states emit the values we observe.
> * __Build a jump hidden Markov model of one asset:__ Bin the daily growth rate into states, estimate the transition matrix by counting, fit an emission distribution in each state, and explain why jump episodes are needed to reproduce volatility clustering.
> * __Compose a synthetic market of many stocks:__ Combine a market draw, a sector draw, and a stock's own draw into synthetic stocks that move together like real ones.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * A Markov chain is a set of states $\mathcal{S}$ and a transition matrix $\mathbf{P}$ whose rows sum to one. Its stationary distribution satisfies $\bar{\pi} = \bar{\pi}\mathbf{P}$.
> * The jump hidden Markov model bins the daily growth rate into states of equal probability, estimates $\mathbf{P}$ by counting, and draws each day's growth rate from a Student-t distribution of the day's state. Jump episodes, which start with probability $\epsilon$ and last $\lambda$ days on average, keep volatility clustering.
> * A synthetic stock is $g_{i} = \alpha_{i} + \beta_{i}\,g_{m} + \delta_{i}\,f_{s} + \psi_{i}\,\tilde{\varepsilon}_{i}$: the market and sector draws it shares with other stocks make them move together.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The power iteration algorithm, the tuning objective for $\epsilon$ and $\lambda$, and how the sector loadings and the own-draw scale are chosen.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13a) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L13a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024, simulate it, and compare its distribution and volatility clustering with the data. We then tune jump episodes and see how much of the clustering they restore.

> [▶ Build a synthetic market of many stocks](CHEME-5660-L13a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.

___

## Why Synthetic Markets?
Previously, we tested trading methods by backtesting them on historical prices. A backtest is a single path: the market went through one sequence of booms and sell-offs, and a method that did well on that sequence may have been lucky. To judge a method, we would like to run it on many plausible paths, including bad ones that did not happen.

A __synthetic market__ is a model that generates such paths. To be useful for stress tests, its paths should share the features of real growth rates that matter for risk:
* __Fat tails:__ large daily moves are far more common than a normal distribution predicts.
* __Volatility clustering:__ large moves follow large moves, so the absolute growth rate stays correlated for weeks.
* __Co-movement:__ stocks move together, through the market and through their sectors.

Geometric Brownian motion, which we used before, can have co-movement through correlated shocks. It has neither fat tails nor volatility clustering, because its daily moves are normally distributed and its volatility is constant. In this lecture, we build a generator that has all three, starting from Markov chains.
___

## Markov Chains
A discrete Markov model is a stochastic (random) process that describes the transitions between states based on defined probabilities. Formally, a Markov model is determined by the tuple $\mathcal{M} = (\mathcal{S},\mathbf{P})$, where $\mathcal{S}$ represents the set of states in the model and $\mathbf{P}$ is the transition matrix.

### The state space $\mathcal{S}$
The state space $\mathcal{S}$ is the set of all possible values a system can assume. For example, if a Markov chain can be in the states $\left\{1,2,3\right\}$, then $\mathcal{S} \equiv \left\{1,2,3\right\}$. What do these states represent? The answer depends on the problem. Here are two concrete examples:
* __Letters in words__ $\mathcal{S} \equiv \left\{a,b,c,\dotsc,z\right\}$: If the state space $\mathcal{S}$ is defined as the alphabet, we could develop a Markov model to generate $n$ words, each of length $l$ characters, that start with a specific letter such as $t$.
* __Investor moods__ $\mathcal{S} \equiv \left\{\text{bullish},\text{neutral},\text{bearish}\right\}$: In this case, the state space is defined as three possible moods an investor can experience. Using this type of $\mathcal{S}$, we could build a Markov model to simulate how an investor's mood changes as they watch the market, the news, and other factors.

Next, let's look at the Markov property and the transition matrix $\mathbf{P}$.

### The Markov Property
The Markov property describes a random process whose next-state probabilities depend only on its present state. Choosing a state that contains the information needed to predict the next step lets us describe the process using one-step transition probabilities.

> __Definition:__ Let $X_n$ denote the state of a stochastic process at time step $n$. The process satisfies the **Markov property** if the following equality holds for every $n\geq 0$ and every state history with positive probability:
>
> $$
> P(X_{n+1}=s_{n+1}\mid X_0=s_0,\ldots,X_n=s_n)
> =P(X_{n+1}=s_{n+1}\mid X_n=s_n).
> $$
>
> Given the present state, the next state is conditionally independent of the past states. This lets us update the state distribution without storing the full history of the process.

#### One-Step Transition Probabilities
For a finite state space $\mathcal{S}=\{s_1,\ldots,s_N\}$, the probability of moving from state $s_i$ to state $s_j$ in one step is given by:

$$
p_{ij}=P(X_{n+1}=s_j\mid X_n=s_i).
$$

We assume the process is __time-homogeneous__, meaning these probabilities stay constant over time. Thus, $p_{ij}$ does not depend on $n$, and the same transition matrix applies at every step.

#### Why the Markov Property Matters
The Markov property simplifies our modeling work: instead of tracking the entire history of a process, we only need to know the present state. Without this property, the number of possible historical sequences would grow exponentially. With it, the problem becomes tractable and easier to work with.

### The Transition Matrix
We collect the one-step probabilities in the transition matrix $\mathbf{P}\in\mathbb{R}^{N\times N}$. Row $i$ describes the next-state distribution when the current state is $s_i$:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-TransitionMatrix-Schematic.svg" width="680" style="max-width:100%; height:auto;" alt="Schematic of a transition matrix drawn as a grid of dots inside square brackets. A shaded row is labeled rows equal current states, with the note sum equals 1. A shaded column is labeled columns equal future states. A red dot in row one, column two is labeled p12, and a yellow dot on the diagonal is labeled pii, a self-loop."/>
    </center>
</div>

The transition matrix has the following properties:
* Every entry is non-negative: $p_{ij}\geq 0$.
* Rows represent the current states, and columns represent the next states (our convention). Entry $p_{ij}$ is the probability of moving from $s_i$ to $s_j$.
* Every row sums to one: $\sum_{j=1}^{N}p_{ij}=1$. Starting from any state, the chain must move to one of the states in $\mathcal{S}$, including the possibility of staying where it is.

> __Example:__ Consider a 2-state Markov model of the weather with states $\mathcal{S} = \{\text{Sunny}, \text{Rainy}\}$ and the transition matrix:
>
> $$\mathbf{P} = \begin{bmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{bmatrix}$$
>
> Row 1 is Sunny and row 2 is Rainy. If today is Sunny, there is an 80% chance tomorrow is Sunny and a 20% chance it is Rainy. If today is Rainy, there is a 40% chance tomorrow is Sunny and a 60% chance it is Rainy.
>
> Starting from the initial distribution $\pi_0 = [1, 0]$, which says today is Sunny, the distribution for tomorrow is:
>
> $$\pi_1 = \pi_0 \mathbf{P} = [1, 0]\begin{bmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{bmatrix} = [0.8, 0.2]$$

#### Estimating the transition matrix from data
In finance, we estimate $\mathbf{P}$ from a sequence of observed states $X_{1},\dotsc,X_{T}$. Let $n_{ij}$ be the number of days in state $s_{i}$ followed by a day in state $s_{j}$. The estimate of $p_{ij}$ is the share of the days in state $s_{i}$ that moved to state $s_{j}$:
$$
\hat{p}_{ij} = \frac{n_{ij}}{\sum_{k=1}^{N} n_{ik}}
$$
Each row of $\hat{\mathbf{P}}$ sums to one by construction. This counting estimate is the maximum likelihood estimate of $\mathbf{P}$, and we will use it to estimate the transition matrix of the market generator.

### The Stationary Distribution
The stationary distribution $\bar{\pi}$ of a Markov chain with transition matrix $\mathbf{P}$ is a probability distribution that remains unchanged as the system evolves: $\bar{\pi} = \bar{\pi}\mathbf{P}$. It is the long-run share of time the chain spends in each state, and it is where a generator starts its paths.

Software often computes $\bar{\pi}$ by solving the linear equations $\bar{\pi} = \bar{\pi}\mathbf{P}$ and $\sum_{i}\bar{\pi}_{i} = 1$ directly. The power iteration method instead applies the transition matrix to an initial distribution until the distribution stops changing:

$$
\begin{align*}
\pi_1 &= \pi_0 \mathbf{P} \\
\pi_2 &= \pi_1 \mathbf{P} = \pi_0 \mathbf{P}^2 \\
\pi_3 &= \pi_2 \mathbf{P} = \pi_0 \mathbf{P}^3 \\
&\vdots \\
\pi_k &= \pi_{k-1} \mathbf{P} = \pi_0 \mathbf{P}^k
\end{align*}
$$

__Initialize__: Given an ergodic transition matrix $\mathbf{P}$ with $N$ states, a tolerance $\text{tol}>0$, and a maximum number of iterations $T$, initialize the iteration counter $t\gets 1$, the uniform initial distribution $\pi \gets \left[1/N,\ldots,1/N\right]$, and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:
1. Compute: $\pi^{\prime} \gets \pi\mathbf{P}$.
2. Check whether to stop:
    - If $\lVert \pi^{\prime} - \pi \rVert_{1} \leq \text{tol}$ __then__ set $\texttt{converged}\gets\texttt{true}$ and return $\bar{\pi}\gets\pi^{\prime}$.
    - Otherwise, if $t\geq{T}$ __then__ return $\pi^{\prime}$ with $\texttt{converged}=\texttt{false}$ and a warning that the maximum number of iterations was reached without convergence.
3. Update the distribution: $\pi\gets\pi^{\prime}$.
4. Increment the iteration counter: $t\gets{t+1}$.

__Output__: The stationary distribution $\bar{\pi}$.

The convergence rate depends on the spectral gap (difference between the largest and second-largest eigenvalue magnitudes of $\mathbf{P}$), with larger gaps enabling faster convergence.

__Notes:__
1. The algorithm assumes $\mathbf{P}$ is ergodic (irreducible and aperiodic). Such a chain has exactly one stationary distribution, and power iteration reaches it from any starting distribution.
2. The $\pi^{\prime} \gets \pi\mathbf{P}$ step is a left vector-matrix multiplication: $\pi$ is a row vector, and the new probability of state $s_{j}$ is $\pi^{\prime}_{j} = \sum_{i}\pi_{i}p_{ij}$.
___

## Hidden Markov Models (HMMs)
Next, let's explore hidden Markov models (HMMs).

> __Definition:__
>
> In a discrete hidden Markov model (HMM), the states are not directly observable. Instead, we observe outputs (emissions) whose probabilities depend on the hidden state. An HMM is characterized by the tuple $\mathcal{M} = (\mathcal{S},\mathcal{O},\mathbf{P},\mathbf{E})$:
> * __State Space ($\mathcal{S}$)__: The set of the $|\mathcal{S}|$ hidden states that the system can be in.
> * __Observation Space ($\mathcal{O}$)__: The set of the $|\mathcal{O}|$ observable states (emissions).
> * __Transition Matrix ($\mathbf{P}$)__: The matrix $\mathbf{P}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{S}|}$ of the probabilities of moving between hidden states.
> * __Emission Matrix ($\mathbf{E}$)__: The matrix $\mathbf{E}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{O}|}$ of the probabilities of observing each observable state given a hidden state.

For every hidden state $s_{i}$, the transition probabilities and the emission probabilities each sum to one:
$$
\begin{align*}
p_{ij} &= P(X_{n+1} = s_j \mid X_{n} = s_i), & \sum_{j=1}^{|\mathcal{S}|} p_{ij} &= 1\\
e_{ij} &= P(O_{n} = o_j \mid X_{n} = s_i), & \sum_{j=1}^{|\mathcal{O}|} e_{ij} &= 1
\end{align*}
$$

### Sampling Algorithm
Let's outline a simple sampling algorithm for generating random sequences from an HMM.

__Initialize__: Given an HMM defined by $\mathcal{M} = (\mathcal{S},\mathcal{O},\mathbf{P},\mathbf{E})$, the number of time steps $T$, and the stationary distribution $\bar{\pi}$. Draw an initial hidden state $X_0$ from the initial distribution: $X_0\sim \bar{\pi}$.

For $n = 1$ to $T$ __do__:
1. Sample the next __hidden state__ $X_n$ from the $\texttt{categorical}$ distribution $\mathbb{P}_{H}(...)$ defined by the previous state $X_{n-1}$ and the transition matrix $\mathbf{P}$: $X_n \sim \mathbb{P}_{H}(X_n \mid X_{n-1})$
2. Sample the observable state $O_n$ from the $\texttt{categorical}$  distribution $\mathbb{P}_{E}(...)$ defined by the current hidden state $X_n$ and the emission matrix $\mathbf{E}$: $O_n \sim \mathbb{P}_{E}(O_n \mid X_n)$
3. Store $X_n$ and $O_n$.

__Output__: The sequences of hidden states $\{X_1, X_2, \ldots, X_T\}$ and observable states $\{O_1, O_2, \ldots, O_T\}$. The initial state $X_0$ is used to generate the sequence but is not included in the output.

> __What is a categorical distribution?__ A categorical distribution describes a random variable that takes one of $k$ categories, each with its own probability. It generalizes the Bernoulli distribution, which has two categories. Each row of $\mathbf{P}$ and of $\mathbf{E}$ is a categorical distribution, and the algorithm samples from these rows at each time step.

#### Continuous emissions
For a market generator, the observation is a growth rate, a real number rather than one of a few categories. In place of a row of $\mathbf{E}$, each hidden state $s_{k}$ has an emission __distribution__, a probability density of the growth rate given the state. Step 2 of the sampling algorithm then draws the growth rate from the emission distribution of the current state:
$$
g_{n} \mid X_{n} = s_{k} \;\sim\; \text{emission distribution of } s_{k}
$$
The hidden state sets how volatile the market is today, and the emission turns the state into a growth rate.
___

## The Jump Hidden Markov Model
Let's build a hidden Markov model of the daily growth rate $g_{t} = \frac{1}{\Delta t}\ln\left(S_{t}/S_{t-1}\right)$ (units: 1/yr) of one asset, following Alswaidan and Varner, who call it a jump hidden Markov model. It has three parts:
* __States:__ We fit a Laplace distribution to the growth rates and use its cumulative distribution function $F_{L}$ to cut it into $N$ slices of equal probability. Each slice is a hidden state, from state 1 with the most negative growth rates to state $N$ with the most positive. In the data, we see each day's state, but in simulation the states are hidden, and only their growth rates are reported.
* __Transition matrix:__ Each day of the data falls in one slice, which gives a sequence of states. We estimate $\hat{\mathbf{P}}$ by counting the moves between the states of consecutive days.
* __Emissions:__ In each state $s_{k}$, the growth rate is drawn from a Student-t distribution, $g_{t} \mid X_{t} = s_{k} \sim \mu_{k} + \sigma_{k}\,T_{\nu}$. The mean $\mu_{k}$ and the standard deviation $\sigma_{k}$ come from the growth rates of the days in state $s_{k}$, and $T_{\nu}$ has $\nu$ degrees of freedom. With $\nu = 5$, the draws in a state spread about 1.3 times $\sigma_{k}$.

The schematic below shows the fit with $N = 5$ states. Each state is a mood of the market, from very bearish (state 1) to very bullish (state 5):

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-JumpHMM-Fit-Schematic.svg" width="1000" style="max-width:100%; height:auto;" alt="The fit of the model in three parts. On the left, a plot of the daily growth rate of a ticker on sixteen trading days. In the middle, on the same growth-rate axis, the fitted Laplace cumulative distribution, with the cumulative probability on the horizontal axis. Vertical dashed lines at 0.2, 0.4, 0.6, and 0.8 cut the probability into N = 5 slices of equal probability, and from the point where each meets the curve a dashed line runs across both plots as a state boundary. A bar to the right of the curve names the state and the mood of each band: state 1, very bearish, in red at the bottom, then state 2, bearish, state 3, neutral, state 4, bullish, and state 5, very bullish, in blue at the top. Above the growth-rate plot, a row gives the state of each day: 3, 4, 3, 2, 3, 4, 5, 1, 5, 1, 2, 3, 4, 3, 3, 2. The three pairs 3, 4 are boxed in orange, and an orange arrow labeled count every move between consecutive days, such as 3 to 4, leads to a table of counts n i j on the right. The table has the state on day t in the rows and the state on day t plus 1 in the columns, and the cell in row 3 and column 4 holds 3 and is highlighted in orange. An arrow leads from the table to a box with the estimate of p i j, n i j divided by the sum of row i, and the example for row 3, 3 over 6."/>
    </center>
</div>

### Fitting algorithm
The algorithm below fits the model to the daily growth rates of one ticker.

__Initialize__: Given the daily growth rates $g_{1},\dotsc,g_{T}$ of one ticker, the number of states $N$, and the degrees of freedom $\nu$ of the emissions. Set the counts $n_{ij}\gets 0$ for every pair of states $s_{i}$ and $s_{j}$.

1. Fit the Laplace distribution by maximum likelihood: set the location $\mu$ to the median of the growth rates, and set the scale $b \gets \frac{1}{T}\sum_{t=1}^{T}\lvert g_{t} - \mu\rvert$.
2. For $t = 1$ to $T$ __do__: compute the cumulative probability $F_{L}(g_{t})$ and find the slice that holds it. Set $X_{t}\gets s_{k}$, where $(k-1)/N \leq F_{L}(g_{t}) < k/N$.
3. For $t = 2$ to $T$ __do__: count the move from the state of day $t-1$ to the state of day $t$. Set $n_{ij}\gets n_{ij} + 1$, where $X_{t-1} = s_{i}$ and $X_{t} = s_{j}$.
4. For each state $s_{i}$ __do__: divide the counts in row $i$ by their sum. Set $\hat{p}_{ij}\gets n_{ij}/\sum_{k=1}^{N} n_{ik}$ for $j = 1,\dotsc,N$.
5. For each state $s_{k}$ __do__: set $\mu_{k}$ to the mean and $\sigma_{k}$ to the standard deviation of the growth rates of the days with $X_{t} = s_{k}$.
6. Compute the stationary distribution $\bar{\pi}$ of $\hat{\mathbf{P}}$.

__Output__: The Laplace location $\mu$ and scale $b$, the transition matrix $\hat{\mathbf{P}}$, the emission parameters $\mu_{k}$ and $\sigma_{k}$ of each state, and the stationary distribution $\bar{\pi}$.

__Notes:__
1. The Laplace cumulative distribution function has a closed form: $F_{L}(g) = \frac{1}{2}\exp\left((g-\mu)/b\right)$ for $g < \mu$, and $F_{L}(g) = 1 - \frac{1}{2}\exp\left(-(g-\mu)/b\right)$ for $g \geq \mu$.
2. Step 2 gives the same states as the slice edges $F_{L}^{-1}(k/N)$: a growth rate between two neighboring edges has a cumulative probability between $(k-1)/N$ and $k/N$.
3. `JumpHMM.jl` handles two rare cases. A state with no counted moves out of it gets the uniform row $\hat{p}_{ij} = 1/N$. A state with fewer than two days uses the mean and the standard deviation of all the growth rates.

The Student-t emissions and the wide extreme slices give the generator fat tails. To simulate a path, we draw a starting state from $\bar{\pi}$, step the chain with $\hat{\mathbf{P}}$, and draw each day's growth rate from the emission of the day's state.

### Why add jumps?
The counted chain forgets quickly. After a volatile day, the next few transitions usually take the chain from an extreme state back to the middle states. The absolute growth rate of a simulated path stays correlated for only a few days, while in real markets volatile periods last for weeks.

The jump hidden Markov model adds __jump episodes__ to keep volatile periods going. On a day with no episode under way, an episode starts with probability $\epsilon$ and lasts a Poisson number of days with mean $\lambda$. Each episode day draws its state from the most negative states with probability $p_{\text{neg}}$, slightly above one half, and from the most positive states otherwise.

An episode is a run of large moves of both signs, not a single price jump. The schematic below puts the pieces together for one simulated day:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13a-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

### Sampling algorithm
The algorithm below draws one path of $T$ days from the fitted model with jump episodes.

__Initialize__: Given the transition matrix $\hat{\mathbf{P}}$, the emission parameters $\mu_{k}$ and $\sigma_{k}$ of each state, the degrees of freedom $\nu$, the stationary distribution $\bar{\pi}$, the episode probability $\epsilon$, the mean episode length $\lambda$, the number of tail states $N_{\text{tail}}$, the probability $p_{\text{neg}}$ that an episode day is negative, and the number of days $T$. The most negative states are $\mathcal{S}_{-} = \left\{s_{1},\dotsc,s_{N_{\text{tail}}}\right\}$, and the most positive states are $\mathcal{S}_{+} = \left\{s_{N-N_{\text{tail}}+1},\dotsc,s_{N}\right\}$. Draw the first state $X_{1}\sim\bar{\pi}$ and its growth rate $g_{1}$ from the emission of $X_{1}$. Set the number of episode days left $\texttt{remaining}\gets 0$.

For $t = 2$ to $T$ __do__:
1. Check for a new episode: if $\texttt{remaining} = 0$, draw $u\sim\text{Uniform}(0,1)$. If $u < \epsilon$, an episode starts, and we draw its length $\texttt{remaining}\sim\text{Poisson}(\lambda)$.
2. Choose the state of day $t$:
    - If $\texttt{remaining} > 0$, day $t$ is an episode day. Draw $v\sim\text{Uniform}(0,1)$. If $v < p_{\text{neg}}$, draw $X_{t}$ uniformly from $\mathcal{S}_{-}$, otherwise draw $X_{t}$ uniformly from $\mathcal{S}_{+}$. Set $\texttt{remaining}\gets\texttt{remaining} - 1$.
    - If $\texttt{remaining} = 0$, day $t$ is an ordinary day. Draw $X_{t}$ from the $\texttt{categorical}$ distribution defined by row $X_{t-1}$ of $\hat{\mathbf{P}}$.
3. Draw the growth rate from the emission of the day's state: $g_{t}\gets\mu_{k} + \sigma_{k}\,T_{\nu}$, where $X_{t} = s_{k}$ and $T_{\nu}$ is a new Student-t draw.
4. Store $X_{t}$ and $g_{t}$.

__Output__: The states $\{X_{1},\dotsc,X_{T}\}$ and the growth rates $\{g_{1},\dotsc,g_{T}\}$ of one path.

__Notes:__
1. The second case of step 2 is the sampling step of the hidden Markov model above. With $\epsilon = 0$ no episode starts, and the algorithm samples a hidden Markov model with no jumps.
2. A Poisson draw can be zero. In that case no episode starts, and day $t$ is an ordinary day.
3. The check in step 1 runs again on the day after an episode ends, so a new episode can follow at once.
4. `JumpHMM.jl` sets $N_{\text{tail}} = 5$ and $p_{\text{neg}} = 0.52$ unless we choose other values.


### Tuning and checking the model
We choose $\epsilon$ and $\lambda$ by a grid search. For each pair, we simulate paths and keep the $M$ paths that contain an episode. The score $J$ measures how far each path's autocorrelation of $|g_{t}|$ at lags 1 to 25 and its kurtosis are from the data's, averaged over the paths:
$$
J(\epsilon,\lambda) = \frac{1}{M}\sum_{m=1}^{M}\left[\sum_{k=1}^{25}\left(\rho_{k} - \hat{\rho}_{m,k}\right)^{2} + 0.2\left(\kappa - \hat{\kappa}_{m}\right)^{2}\right]
$$
where $\rho_{k}$ and $\kappa$ are the autocorrelation of $|g_{t}|$ at lag $k$ and the kurtosis of the data, and $\hat{\rho}_{m,k}$ and $\hat{\kappa}_{m}$ are those of simulated path $m$.

We keep the pair with the smallest $J$. Because the score uses a finite number of simulated paths, the chosen pair can change with the random seed. Finally, we check the tuned model against the data: the distribution of each simulated path (a Kolmogorov–Smirnov test), its kurtosis, and its volatility clustering.

> __Example:__
>
> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L13a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024, simulate it, and compare its distribution and volatility clustering with the data. We then tune jump episodes and see how much of the clustering they restore.

Next, let's extend the generator from one asset to many.
___

## Many Assets: The Sector Factor Model
A stress test of a portfolio needs synthetic stocks that move together the way real ones do. Independent jump hidden Markov models, one for each stock, would not move together at all. The __sector factor model__ builds each synthetic stock from a market draw and a sector draw that it shares with other stocks, plus a draw of its own.

The daily growth rate of stock $i$ in sector $s$ is:
$$
g_{i}(t) = \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + \psi_{i}\,\tilde{\varepsilon}_{i}(t)
$$
where:
* The market draw $g_{m}(t)$ comes from a jump hidden Markov model of the market (`SPY`). The coefficients $\alpha_{i}$ and $\beta_{i}$ are the stock's single index model estimates, which we hold fixed.
* The sector draw $f_{s}(t)$ comes from a jump hidden Markov model of the sector factor, the average single index residual of the stocks in sector $s$. The loading $\delta_{i}$ is the slope of the stock's residual on the other residuals in its sector, scaled so that the sector's co-movement matches the data.
* The own draw $\tilde{\varepsilon}_{i}(t)$ comes from a jump hidden Markov model of the stock itself, centered on its long-run mean. Its scale $\psi_{i}$ is chosen so that $\psi_{i}^{2}$ is the share of the stock's variance that the market and sector terms leave unexplained. The synthetic stock's variance then matches its real variance on average.

On each day, every stock uses the same market draw, and every stock in a sector uses the same sector draw. These shared draws are the only link between stocks.

### Sampling Algorithm
The algorithm below draws one synthetic path of the whole universe over $T$ days.

__Initialize__: Given the jump hidden Markov models of the market, of each sector's factor, and of each stock; each stock's single index estimates $\alpha_{i}$ and $\beta_{i}$ and sector loading $\delta_{i}$; the variances $\operatorname{Var}(g_{m})$, $\operatorname{Var}(f_{s})$, and $\operatorname{Var}(g_{i})$ of the real growth rates; each stock's real starting price $S_{i}(0)$; and the number of days $T$. Draw a market path $g_{m}(1),\dotsc,g_{m}(T)$ from the market model. For each sector $s$, draw a factor path $f_{s}(1),\dotsc,f_{s}(T)$ from the model of sector $s$, independently of the other sectors and of the market.

For each stock $i$ in sector $s$ __do__:
1. Draw a path of the stock's own model and subtract the model's long-run mean (the mean of one long simulated path) to get $\tilde{\varepsilon}_{i}(1),\dotsc,\tilde{\varepsilon}_{i}(T)$.
2. Choose the scale of the own draw from the share of the stock's real variance that the market and sector terms explain:
    - If $\beta_{i}^{2}\operatorname{Var}(g_{m}) + \delta_{i}^{2}\operatorname{Var}(f_{s}) \leq 0.9\operatorname{Var}(g_{i})$, __then__ set $\psi_{i}^{2} \gets 1 - \left(\beta_{i}^{2}\operatorname{Var}(g_{m}) + \delta_{i}^{2}\operatorname{Var}(f_{s})\right)/\operatorname{Var}(g_{i})$.
    - If $\beta_{i}^{2}\operatorname{Var}(g_{m}) + \delta_{i}^{2}\operatorname{Var}(f_{s}) > 0.9\operatorname{Var}(g_{i})$, __then__ multiply $\beta_{i}$ and $\delta_{i}$ by $\sqrt{0.9\operatorname{Var}(g_{i})/\left(\beta_{i}^{2}\operatorname{Var}(g_{m}) + \delta_{i}^{2}\operatorname{Var}(f_{s})\right)}$ for the whole path, so the two terms explain 90% of the variance, and set $\psi_{i}^{2} \gets 0.1$.
3. Compose the growth rate for $t = 1,\dotsc,T$: $g_{i}(t) \gets \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + \psi_{i}\,\tilde{\varepsilon}_{i}(t)$.
4. Compound the price for $t = 1,\dotsc,T$, starting from $S_{i}(0)$: $S_{i}(t) \gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$.

__Output__: The market path $g_{m}(t)$, and the growth rates $g_{i}(t)$ and prices $S_{i}(t)$ of every synthetic stock, for $t = 1,\dotsc,T$.

__Notes:__
1. `JumpHMM.jl` checks one case before step 2: for a stock whose single index $R^{2}$ is at least 0.80, it keeps both loadings and sets $\psi_{i}^{2}$ so the synthetic stock keeps that $R^{2}$. No stock in the many-stock example reaches 0.80, so step 2 applies to all of them.
2. With the drifting β option below, $\beta_{i}$ becomes $\beta_{i}(t)$ in step 3, and $\beta_{i}^{2}$ in step 2 becomes the long-run mean of $\beta_{i}(t)^{2}$.

### What the model keeps and what it gives up
Every draw comes from a jump hidden Markov model, so each synthetic stock has fat tails and clustered volatility. Its tails and clustering are those of the sum of its three draws, rather than an exact copy of the real stock's.

The stocks in a sector move together through their shared sector draw, and all stocks move together through the market draw. The sector draws are independent of each other. Two stocks in different sectors are therefore linked only through the market.

__Why not a copula?__ A copula imposes a fitted correlation by reordering each stock's draws across days. The reordering would scramble the timing of the volatile periods and jump episodes in those draws. Composing stocks from shared draws on the same day leaves that timing as it was simulated.

### An option: a drifting β
Real β wanders over time, but the model above holds $\beta_{i}$ fixed. The model can instead let $\beta_{i}(t)$ drift slowly around $\hat{\beta}_{i}$ as an exponential moving average of random shocks. The shocks are sized by how much the online estimate of β wanders in the real data.

A trading method that re-estimates β as it trades then faces the same problem in the synthetic market as in the real one.

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L13a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.
___

## Summary
This lecture builds synthetic markets for stress tests, from Markov chains and hidden Markov models to a jump hidden Markov model of one asset and a sector factor model of many stocks.

> __Key Takeaways:__
>
> * __Markov chains and hidden Markov models describe sequences with memory of the present state:__ We defined the state space, the transition matrix, and the stationary distribution, and we estimated the transition matrix by counting transitions. Hidden states extend the chain, and their emission distributions produce the values we observe.
> * __The jump hidden Markov model reproduces fat tails and part of volatility clustering:__ We binned the daily growth rate into states of equal probability, counted transitions, and drew growth rates from a Student-t distribution in each state. Because the counted chain forgets volatile days quickly, we added jump episodes and tuned their probability and length against the data.
> * __Shared draws make synthetic stocks move together:__ We composed each synthetic stock from a market draw and a sector draw that it shares with other stocks, plus a draw of its own. We scaled the own draw so that the stock's variance matches its real variance on average.

Synthetic markets let us test a trading method on many plausible paths, including sell-offs that never happened, rather than on one historical path.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___